# 02 — Prétraitement

1. Recharger les données brutes
2. Rééquilibrer les classes si nécessaire (sous-échantillonnage de la classe majoritaire)
3. Encoder les étiquettes textuelles en format numérique (LabelEncoder + to_categorical)
4. Diviser en ensembles d'entraînement et de test (stratifié, seed fixe)
5. Normaliser les pixels dans la plage [0, 1] (à l'enregistrement, par morceaux : RAM constante)
6. Sauvegarder les données traitées dans `data/processed/` (.npy + class_names.json)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from src import load_data, rebalance_data, preprocess_data, save_processed_data
from src.config import PROCESSED_DATA_DIR

I0000 00:00:1793181960.978356   10199 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793181961.393694   10199 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793181962.935910   10199 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
x, y, class_names = load_data()
x, y = rebalance_data(x, y)

Loaded 4737 images across 4 classes
X shape: (4737, 224, 224, 3), y shape: (4737,)
Rebalanced to 711 images per class (2844 total)


In [4]:
dataset = preprocess_data(x, y)

Encoded labels: {np.str_('Glioma'): 0, np.str_('Meningioma'): 1, np.str_('No Tumor'): 2, np.str_('Pituitary'): 3}
Train: (2275, 224, 224, 3), Test: (569, 224, 224, 3)


In [ ]:
# --- Data augmentation (train only) ---
import tensorflow as tf

def add_noise(image):
    noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=0.05)
    return tf.clip_by_value(image + noise, 0.0, 1.0)

def add_blur(image):
    # Simple average blur (3x3 kernel)
    image = tf.expand_dims(image, 0)  # add batch dim
    image = tf.nn.avg_pool(image, ksize=3, strides=1, padding='SAME')
    return tf.squeeze(image, 0)

def augment(image):
    image = tf.image.random_flip_left_right(image)
    image = add_noise(image)
    image = add_blur(image)
    return image

# Apply augmentation to the training set
x_train = dataset['x_train'].astype('float32') / 255.0
x_train_aug = tf.map_fn(augment, x_train)
# Concatenate original and augmented data
x_train_aug_np = x_train_aug.numpy()
dataset['x_train'] = tf.concat([x_train, x_train_aug_np], axis=0).numpy()
dataset['y_train'] = tf.concat([dataset['y_train'], dataset['y_train']], axis=0).numpy()
print('Training data augmented: ', dataset['x_train'].shape)

E0000 00:00:1793181969.988538   10199 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [ ]:
save_processed_data(dataset, PROCESSED_DATA_DIR)

Processed data saved to /home/brahim-44/Desktop/brainscan-ai/data/processed


In [ ]:
# Visualize class distribution and sample images
from src import show_sample_images
from src.visualization import plot_class_distribution
plot_class_distribution(y)
show_sample_images(x, y, class_names, samples_per_class=5)

In [ ]:
# Visualize class distribution and sample images (fixed)
from src import show_sample_images, count_images_per_class
from src.visualization import plot_class_distribution
from src.config import REPORTS_FIGURES_DIR
counts = count_images_per_class(y)
plot_class_distribution(counts, REPORTS_FIGURES_DIR)
show_sample_images(x, y, class_names, samples_per_class=5)